In [ ]:
# ============================================================
# 1. Google Drive 연결
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# 1-1. 필요한 라이브러리 준비
# ============================================================

# iPhone에서 촬영한 HEIC/HEIF 이미지 지원
!pip -q install pillow-heif

import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# HEIC/HEIF 이미지를 Pillow에서 열 수 있도록 등록
register_heif_opener()

print("라이브러리 준비 완료")
print("TensorFlow version:", tf.__version__)

.

.

.
# ★★★★ (1) 파일 경로, 클래스, 파일 이름 등 설정

In [ ]:
# ============================================================
# 2. 데이터 경로 및 기본 설정
# ============================================================

# Google Drive에 만든 프로젝트 폴더 경로
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Project"

# 학습 이미지와 테스트 이미지가 저장된 폴더
TRAIN_DIR = f"{PROJECT_DIR}/Training"
TEST_DIR = f"{PROJECT_DIR}/Test"

# 실제 프로젝트에 맞게 클래스 이름을 수정할 수 있습니다.
# 파일명은 계속 Class 1, Class 2, Class 3 형식을 사용합니다.
CLASS_NAMES = [
    "Class 1",
    "Class 2",
    "Class 3"
]

# CNN에 입력할 이미지 크기
# 원본 이미지는 정사각형으로 자른 후 이 크기로 축소됩니다.
IMAGE_SIZE = 96

# 이번 실험에서 사용할 클래스별 학습 이미지 수
# 데이터 크기 비교 시 10, 30, 50으로 변경합니다.
# None이면 test 폴더의 모든 이미지를 사용합니다.
TRAIN_SAMPLES_PER_CLASS = None
TEST_SAMPLES_PER_CLASS = None

# 학습 데이터 중 validation에 사용할 비율
# 예: 0.2는 선택된 학습 이미지의 20%를 validation에 사용
VALIDATION_RATIO = 0.2

# 같은 조건에서 같은 이미지들이 선택되도록 하는 난수값
RANDOM_SEED = 42

print("Training folder:", TRAIN_DIR)
print("Test folder:", TEST_DIR)
print("Selected training samples per class:", TRAIN_SAMPLES_PER_CLASS)

In [ ]:
# ============================================================
# 3. Training/Test 폴더의 클래스별 이미지 수 확인
# ============================================================

NUM_CLASSES = len(CLASS_NAMES)

# 사용할 이미지 확장자
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp",
    ".heic", ".heif"
}


def collect_files_by_class(folder_path):
    """
    파일명의 Class 1, Class 2, Class 3 부분을 읽어서
    클래스별로 이미지 파일을 분류합니다.
    """

    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(
            f"폴더를 찾을 수 없습니다: {folder_path}"
        )

    files_by_class = {
        class_index: []
        for class_index in range(NUM_CLASSES)
    }

    unrecognized_files = []

    for file_path in sorted(folder.iterdir()):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        # 파일명 시작 부분에서 Class 번호를 찾음
        # 예: Class 2 (15).jpg → class_number = 2
        match = re.match(
            r"^Class\s*(\d+)",
            file_path.stem,
            flags=re.IGNORECASE
        )

        if match is None:
            unrecognized_files.append(file_path.name)
            continue

        class_number = int(match.group(1))
        class_index = class_number - 1

        if class_index not in files_by_class:
            unrecognized_files.append(file_path.name)
            continue

        files_by_class[class_index].append(str(file_path))

    return files_by_class, unrecognized_files


# Training과 Test 폴더 검사
train_files_all, train_unrecognized = collect_files_by_class(TRAIN_DIR)
test_files_all, test_unrecognized = collect_files_by_class(TEST_DIR)


def print_file_counts(title, files_by_class):
    print(f"\n{title}")
    print("-" * 40)

    total_count = 0

    for class_index, class_name in enumerate(CLASS_NAMES):
        count = len(files_by_class[class_index])
        total_count += count

        print(f"{class_name}: {count}개")

    print("-" * 40)
    print(f"Total: {total_count}개")


print_file_counts(
    "Training folder image counts",
    train_files_all
)

print_file_counts(
    "Test folder image counts",
    test_files_all
)


# 파일명 형식이 맞지 않는 이미지가 있으면 출력
if train_unrecognized:
    print("\nTraining 폴더에서 클래스가 인식되지 않은 파일:")
    for filename in train_unrecognized:
        print(" -", filename)

if test_unrecognized:
    print("\nTest 폴더에서 클래스가 인식되지 않은 파일:")
    for filename in test_unrecognized:
        print(" -", filename)

In [ ]:
# ============================================================
# 4. 이미지 선택 및 전처리
# ============================================================

def select_files_per_class(
    files_by_class,
    samples_per_class,
    random_seed
):
    """
    각 클래스에서 지정한 수만큼 이미지를 선택합니다.

    samples_per_class가 None이면 모든 이미지를 사용합니다.
    """

    selected_files = {}

    for class_index in range(NUM_CLASSES):

        class_files = list(files_by_class[class_index])

        # 클래스별로 항상 동일한 순서가 만들어지도록 설정
        rng = np.random.default_rng(
            random_seed + class_index
        )

        random_order = rng.permutation(len(class_files))

        shuffled_files = [
            class_files[index]
            for index in random_order
        ]

        if samples_per_class is None:
            selected_files[class_index] = shuffled_files

        else:
            if len(shuffled_files) < samples_per_class:
                raise ValueError(
                    f"{CLASS_NAMES[class_index]}에 필요한 이미지가 부족합니다.\n"
                    f"필요한 수: {samples_per_class}\n"
                    f"현재 수: {len(shuffled_files)}"
                )

            selected_files[class_index] = (
                shuffled_files[:samples_per_class]
            )

    return selected_files


def load_and_preprocess_images(
    files_by_class,
    image_size
):
    """
    이미지 파일을 불러온 후 다음 작업을 수행합니다.

    1. 사진 회전 정보 보정
    2. RGB 이미지로 변환
    3. 중앙 기준 정사각형 크롭
    4. 지정한 크기로 축소
    5. 픽셀 값을 0~1 범위로 정규화
    """

    image_arrays = []
    labels = []
    file_paths = []

    for class_index in range(NUM_CLASSES):

        for file_path in files_by_class[class_index]:

            try:
                with Image.open(file_path) as image:

                    # 스마트폰 사진의 회전 정보 적용
                    image = ImageOps.exif_transpose(image)

                    # RGB 이미지로 변환
                    image = image.convert("RGB")

                    # 중앙을 기준으로 정사각형 크롭 후 크기 변경
                    image = ImageOps.fit(
                        image,
                        (image_size, image_size),
                        method=Image.Resampling.LANCZOS,
                        centering=(0.5, 0.5)
                    )

                    # NumPy 배열로 변환하고 0~1로 정규화
                    image_array = (
                        np.asarray(image, dtype=np.float32)
                        / 255.0
                    )

                image_arrays.append(image_array)
                labels.append(class_index)
                file_paths.append(file_path)

            except Exception as error:
                print(f"이미지를 읽지 못했습니다: {file_path}")
                print("Error:", error)

    return (
        np.asarray(image_arrays, dtype=np.float32),
        np.asarray(labels, dtype=np.int32),
        np.asarray(file_paths)
    )


# 클래스별로 사용할 학습 이미지 선택
selected_train_files = select_files_per_class(
    train_files_all,
    TRAIN_SAMPLES_PER_CLASS,
    RANDOM_SEED
)

# 테스트 이미지는 기본적으로 모두 사용
selected_test_files = select_files_per_class(
    test_files_all,
    TEST_SAMPLES_PER_CLASS,
    RANDOM_SEED
)

# 이미지 전처리 및 배열 변환
X_train_all, y_train_all, train_file_paths_all = (
    load_and_preprocess_images(
        selected_train_files,
        IMAGE_SIZE
    )
)

X_test, y_test, test_file_paths = (
    load_and_preprocess_images(
        selected_test_files,
        IMAGE_SIZE
    )
)

print("이미지 전처리 완료")
print("Training image array shape:", X_train_all.shape)
print("Test image array shape:", X_test.shape)
print("\n 데이터 배열 형태: (이미지 개수, 이미지 높이, 이미지 너비, RGB 채널 수)")

In [ ]:
# ============================================================
# 5. 레이블 확인 및 Training/Validation 데이터 분리
# ============================================================

# 학습 이미지 일부를 validation 데이터로 분리
(
    X_train,
    X_validation,
    y_train,
    y_validation,
    train_file_paths,
    validation_file_paths
) = train_test_split(
    X_train_all,
    y_train_all,
    train_file_paths_all,
    test_size=VALIDATION_RATIO,
    stratify=y_train_all,
    random_state=RANDOM_SEED
)


def print_label_counts(title, labels):
    print(f"\n{title}")
    print("-" * 40)

    for class_index, class_name in enumerate(CLASS_NAMES):
        count = int(np.sum(labels == class_index))
        print(f"{class_name}: {count}개")


print_label_counts(
    "Actual training data",
    y_train
)

print_label_counts(
    "Validation data",
    y_validation
)

print_label_counts(
    "Independent test data",
    y_test
)

print("\nData shapes")
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("\nLabel mapping")
for class_index, class_name in enumerate(CLASS_NAMES):
    print(f"{class_index} → {class_name}")

.

.

.
# ★★★★ (2) CNN 학습 파라미터 설정

In [ ]:
# ============================================================
# 6-1. 사용자가 수정할 CNN 학습 설정
# ============================================================

# CNN convolution block 수
# 권장 실험값: 1, 2, 3
CONV_BLOCKS = 2

# 첫 번째 convolution layer의 filter 수
# 다음 블록에서는 2배씩 증가합니다.
# 예: 16 → 32 → 64
BASE_FILTERS = 16

# Fully connected layer의 neuron 수
DENSE_UNITS = 64

# Dropout 비율
# 0.30은 neuron의 30%를 무작위로 비활성화
DROPOUT_RATE = 0.30

# Learning rate
# 권장 실험값: 0.0001, 0.001, 0.01
LEARNING_RATE = 0.001

# 전체 학습 반복 횟수
# 권장 실험값: 10, 20, 50
EPOCHS = 50

# 한 번에 모델에 입력하는 이미지 수
# 권장 실험값: 4, 8, 16
BATCH_SIZE = 8

# 이미지 변형을 이용한 data augmentation 사용 여부
# 데이터 크기 비교 실험에서는 False를 권장합니다.
USE_DATA_AUGMENTATION = False

print("CNN 설정 완료")

In [ ]:
# ============================================================
# 6-2. CNN 모델 생성
# ============================================================

# 같은 조건에서는 유사한 결과가 나오도록 설정
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# 이전에 만들어진 모델 정보 초기화
tf.keras.backend.clear_session()

model_layers = [
    layers.Input(
        shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
    )
]

# 선택 사항: 학습 중 이미지에 작은 변형을 적용
if USE_DATA_AUGMENTATION:
    model_layers.extend([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10)
    ])

# 지정한 CONV_BLOCKS 수만큼 convolution block 생성
for block_index in range(CONV_BLOCKS):

    number_of_filters = (
        BASE_FILTERS * (2 ** block_index)
    )

    model_layers.extend([
        layers.Conv2D(
            filters=number_of_filters,
            kernel_size=(3, 3),
            activation="relu",
            padding="same"
        ),

        layers.MaxPooling2D(
            pool_size=(2, 2)
        )
    ])

# CNN에서 추출된 정보를 최종 분류에 사용
model_layers.extend([
    layers.GlobalAveragePooling2D(),

    layers.Dense(
        DENSE_UNITS,
        activation="relu"
    ),

    layers.Dropout(
        DROPOUT_RATE
    ),

    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

model = models.Sequential(model_layers)

# 모델 학습 방법 설정
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

# 모델 구조 출력
model.summary()

In [ ]:
# ============================================================
# 7. CNN 모델 학습
# ============================================================

history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_validation,
        y_validation
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

In [ ]:
# ============================================================
# 7-1. Training Loss와 Validation Loss
# ============================================================

plt.figure(figsize=(7, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 7-2. Training Accuracy와 Validation Accuracy
# ============================================================

plt.figure(figsize=(7, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 8. 독립적인 테스트 이미지 평가
# ============================================================


# 전체 테스트 데이터에 대한 loss와 accuracy 계산
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2%}")


# 각 테스트 이미지에 대한 예측 확률 계산
prediction_probabilities = model.predict(
    X_test,
    verbose=0
)

# 가장 높은 확률을 가진 클래스 선택
predicted_labels = np.argmax(
    prediction_probabilities,
    axis=1
)

# 가장 높은 confidence 값
prediction_confidences = np.max(
    prediction_probabilities,
    axis=1
)


# 결과 표 생성
prediction_results = pd.DataFrame({
    "File Name": [
        Path(file_path).name
        for file_path in test_file_paths
    ],

    "Actual Class": [
        CLASS_NAMES[label]
        for label in y_test
    ],

    "Predicted Class": [
        CLASS_NAMES[label]
        for label in predicted_labels
    ],

    "Confidence (%)": (
        prediction_confidences * 100
    ).round(2),

    "Correct": (
        y_test == predicted_labels
    )
})

# 클래스와 파일명 순서로 정렬
prediction_results = prediction_results.sort_values(
    by=["Actual Class", "File Name"]
).reset_index(drop=True)

pd.set_option("display.max_rows", None)

display(prediction_results)

In [ ]:
# ============================================================
# 9. Confusion Matrix
# ============================================================

confusion_matrix_values = confusion_matrix(
    y_test,
    predicted_labels,
    labels=range(NUM_CLASSES)
)

display_object = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_values,
    display_labels=CLASS_NAMES
)

figure, axis = plt.subplots(figsize=(7, 6))

display_object.plot(
    ax=axis,
    values_format="d",
    colorbar=False
)

plt.title("Confusion Matrix for Test Data")
plt.xticks(rotation=20)
plt.show()

.

.

.
# ★★★★ (3) 모델 저장 경로 설정

In [ ]:
# ============================================================
# 10. 학습된 CNN 모델 저장
# ============================================================

SAVE_DIR = f"{PROJECT_DIR}/Saved_models"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

# 실험 조건이 파일명에 포함되도록 설정
MODEL_FILE_NAME = (
    f"cnn_"
    f"train{TRAIN_SAMPLES_PER_CLASS}_"
    f"size{IMAGE_SIZE}_"
    f"blocks{CONV_BLOCKS}_"
    f"epochs{EPOCHS}.keras"
)

MODEL_SAVE_PATH = (
    f"{SAVE_DIR}/{MODEL_FILE_NAME}"
)

# CNN 모델 저장
model.save(MODEL_SAVE_PATH)

print("Model saved successfully.")
print("Saved path:")
print(MODEL_SAVE_PATH)